# RAGFlow Citation Processing - End-to-End Pipeline

> Reproduce RAGFlow's citation pipeline: **prompt construction -> LLM generation -> backend vector-level validation -> structured output**

## Architecture

```
Phase 1: Retrieve + prompt assembly -> chunks tagged [ID:x] -> citation_prompt -> full prompt
Phase 2: LLM free generation -> full prompt -> LLM -> raw answer (may have missing/wrong/hallucinated citations)
Phase 3: Backend insert_citations() -> split sentences -> embed -> hybrid similarity -> threshold check -> correct tags
Phase 4: Structured JSON output -> parse [ID:x] -> build reference_chunks -> standard API response
```


## Cell 1: Install Dependencies


In [1]:
import sys, subprocess
for pkg in ["openai", "numpy", "jinja2"]:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"[OK] {pkg}")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"[OK] {pkg}")


[OK] openai
[OK] numpy
[OK] jinja2


## Cell 2: Imports & Configuration


In [2]:
import os, re, json, textwrap
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional

USE_MOCK = True  # Set False for real API
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
LLM_API_KEY  = os.getenv("LLM_API_KEY", "ollama")
LLM_MODEL    = os.getenv("LLM_MODEL", "qwen2.5:7b")
EMBED_BASE   = os.getenv("EMBED_BASE", "http://localhost:11434/v1")
EMBED_KEY    = os.getenv("EMBED_KEY", "ollama")
EMBED_MODEL  = os.getenv("EMBED_MODEL", "nomic-embed-text")

PD = Path(__file__).parent if '__file__' in dir() else Path.cwd()
if not (PD / "test_chunks.json").exists():
    PD = Path.cwd()
print(f"Dir: {PD} | Mode: {'Mock' if USE_MOCK else 'Real API'}")


Dir: /workspace/project/notebook/rag_citation | Mode: Mock


## Cell 3: EmbeddingClient & LLMClient


In [3]:
class EmbeddingClient:
    """OpenAI-compatible embedding client with bag-of-words mock."""
    def __init__(self, base_url=None, api_key=None, model=None, use_mock=True):
        self.use_mock = use_mock
        self.model = model or "text-embedding-ada-002"
        self._vocab = {}
        self._vd = 128
        if not use_mock:
            from openai import OpenAI
            self.client = OpenAI(base_url=base_url, api_key=api_key)

    def _wvec(self, w):
        """Get deterministic vector for a word."""
        if w not in self._vocab:
            np.random.seed(hash(w) % 100000)
            self._vocab[w] = np.random.randn(self._vd).astype(np.float32)
        return self._vocab[w]

    def _encode(self, text):
        """Encode text as mean of word vectors (bag-of-words mock)."""
        words = text.lower().split()
        if not words:
            return np.zeros(self._vd, dtype=np.float32)
        v = np.mean([self._wvec(w) for w in words], axis=0)
        n = np.linalg.norm(v)
        return v / n if n > 0 else v

    def encode(self, texts):
        """Encode list of texts to numpy array of vectors."""
        if isinstance(texts, str):
            texts = [texts]
        if self.use_mock:
            return np.array([self._encode(t) for t in texts])
        resp = self.client.embeddings.create(model=self.model, input=texts)
        return np.array([d.embedding for d in resp.data])


class LLMClient:
    """OpenAI-compatible chat client with mock."""
    def __init__(self, base_url=None, api_key=None, model=None, use_mock=True):
        self.use_mock = use_mock
        self.model = model or "gpt-3.5-turbo"
        if not use_mock:
            from openai import OpenAI
            self.client = OpenAI(base_url=base_url, api_key=api_key)

    def chat(self, sys_p, usr_p, max_tokens=2048):
        """Chat with LLM or return mock response."""
        if self.use_mock:
            return self._mock(sys_p, usr_p)
        r = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": sys_p},
                {"role": "user", "content": usr_p}
            ],
            max_tokens=max_tokens, temperature=0.7)
        return r.choices[0].message.content

    def _mock(self, s, u):
        """Return a mock LLM response with intentional citation issues."""
        return (
            "Based on the documentation provided, here is the answer:\n"
            "\n"
            "The IBM Content Navigator Task Manager does not support five key functions. "
            "These include document version control integration [ID:0] and custom workflow "
            "triggers based on document metadata changes. Additionally, real-time collaborative "
            "editing of task descriptions is not supported [ID:2]. The system also lacks "
            "automated task archiving based on retention policies [ID:0], and cross-repository "
            "task delegation is not available.\n"
            "\n"
            "For FileNet deployments, administrators should note that the Content Engine handles "
            "document storage and metadata management [ID:1], while the Process Engine manages "
            "workflow execution [ID:1]. Common issues include database connectivity problems "
            "and JNDI configuration errors."
        )


emb = EmbeddingClient(
    base_url=EMBED_BASE, api_key=EMBED_KEY,
    model=EMBED_MODEL, use_mock=USE_MOCK)
llm = LLMClient(
    base_url=LLM_BASE_URL, api_key=LLM_API_KEY,
    model=LLM_MODEL, use_mock=USE_MOCK)
print("OK: Clients initialized")


OK: Clients initialized


## Cell 4: Load Test Data


In [4]:
def load_chunks(fp=None):
    """Load test chunks from JSON file."""
    if fp is None:
        fp = PD / "test_chunks.json"
    with open(fp, 'r') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} chunks:")
    for ch in data:
        preview = ch['content'][:60]
        print(f"  [ID:{ch['id']}] {ch['doc_name'][:50]} - {preview}...")
    return data

chunks = load_chunks()


Loaded 2 chunks:
  [ID:0] IBM_Content_Navigator_Administration_Guide.pdf - IBM Content Navigator Task Manager provides five key functio...
  [ID:1] FileNet_Deployment_Best_Practices.pdf - FileNet Content Manager deployment requires careful planning...


## Cell 5: kb_prompt() - Knowledge Base Context


In [5]:
def kb_prompt(chunks, max_tok=4096):
    """Format chunks into RAGFlow-style knowledge base context."""
    lines = []
    for ch in chunks:
        lines.append(f"ID: {ch.get('id', 0)}")
        lines.append(f"  Title: {ch.get('doc_name', '?')}")
        if ch.get('page_num'):
            lines.append(f"  Page: {ch['page_num']}")
        lines.append(f"  Content: {ch['content']}")
        lines.append("")
    return "\n".join(lines)

ctx = kb_prompt(chunks)
print("Knowledge Base Context:")
print("-" * 60)
print(ctx[:500] + "...")
print("-" * 60)


Knowledge Base Context:
------------------------------------------------------------
ID: 0
  Title: IBM_Content_Navigator_Administration_Guide.pdf
  Page: 147
  Content: IBM Content Navigator Task Manager provides five key functions: task assignment, task routing, task monitoring, task escalation, and task reporting. However, the following five functions are NOT supported with the Task Manager: (1) document version control integration, (2) custom workflow triggers based on document metadata changes, (3) real-time collaborative editing of task descriptions, (4) automated task arc...
------------------------------------------------------------


## Cell 6: citation_prompt() - Citation Format Rules


In [6]:
CITATION_TPL = (
    "## Citation Guidelines\n"
    "\n"
    "1. Every factual claim MUST be cited as [ID:x] where x is the chunk ID.\n"
    "2. Place citations IMMEDIATELY after the claim, before punctuation.\n"
    "   Correct: No version control [ID:0].\n"
    "   Wrong: No version control. [ID:0]\n"
    "3. Up to 4 citations per sentence allowed.\n"
    "4. ONLY cite chunks that actually support the claim. No invented citations.\n"
    "5. Unsupported claims: state without citation or say info not available.\n"
    "6. No citations for general knowledge or transitional phrases.\n"
)


def citation_prompt():
    """Return citation format template."""
    return CITATION_TPL


print("Citation Prompt Template:")
print("-" * 40)
print(citation_prompt())


Citation Prompt Template:
----------------------------------------
## Citation Guidelines

1. Every factual claim MUST be cited as [ID:x] where x is the chunk ID.
2. Place citations IMMEDIATELY after the claim, before punctuation.
   Correct: No version control [ID:0].
   Wrong: No version control. [ID:0]
3. Up to 4 citations per sentence allowed.
4. ONLY cite chunks that actually support the claim. No invented citations.
5. Unsupported claims: state without citation or say info not available.
6. No citations for general knowledge or transitional phrases.



## Cell 7: build_rag_prompt() - Assemble Full Prompt


In [7]:
def build_rag_prompt(chunks, question):
    """Assemble full System Prompt + User Prompt."""
    sys_p = (
        "You are a helpful assistant answering questions based on provided "
        "knowledge base documents. Use [ID:x] format for citations."
    )
    context = kb_prompt(chunks)
    rules = citation_prompt()
    usr_p = (
        f"## Knowledge Base Context\n\n{context}\n\n"
        f"## Citation Rules\n\n{rules}\n\n"
        f"## Question\n\n{question}\n\n"
        f"Answer based on the context with proper [ID:x] citations."
    )
    return sys_p, usr_p


Q = (
    "Which five functions are not supported by the IBM Content Navigator "
    "Task Manager? Describe FileNet deployment."
)
sys_p, usr_p = build_rag_prompt(chunks, Q)
print(f"System Prompt: {len(sys_p)} chars")
print(f"User Prompt: {len(usr_p)} chars")


System Prompt: 124 chars
User Prompt: 2412 chars


## Cell 8: LLM Raw Answer Generation


In [8]:
raw = llm.chat(sys_p, usr_p)
print("Raw LLM Answer:")
print("=" * 60)
print(raw)
print("=" * 60)
cites = re.findall(r'\[ID:(\d+)\]', raw)
print(f"Citations found: {cites} | Unique: {sorted(set(cites))}")


Raw LLM Answer:
Based on the documentation provided, here is the answer:

The IBM Content Navigator Task Manager does not support five key functions. These include document version control integration [ID:0] and custom workflow triggers based on document metadata changes. Additionally, real-time collaborative editing of task descriptions is not supported [ID:2]. The system also lacks automated task archiving based on retention policies [ID:0], and cross-repository task delegation is not available.

For FileNet deployments, administrators should note that the Content Engine handles document storage and metadata management [ID:1], while the Process Engine manages workflow execution [ID:1]. Common issues include database connectivity problems and JNDI configuration errors.
Citations found: ['0', '2', '0', '1', '1'] | Unique: ['0', '1', '2']


## Cell 9: Analyze Citation Issues


In [9]:
valid_ids = {ch['id'] for ch in chunks}
found = set(int(m) for m in re.findall(r'\[ID:(\d+)\]', raw))
hallucinated = found - valid_ids
valid_found = found & valid_ids
print(f"Valid IDs in chunks: {sorted(valid_ids)}")
print(f"IDs cited by LLM:  {sorted(found)}")
if hallucinated:
    print(f"HALLUCINATED citations: {[f'[ID:{i}]' for i in sorted(hallucinated)]}")
else:
    print("No hallucinated citations")
if valid_found:
    print(f"Valid citations: {[f'[ID:{i}]' for i in sorted(valid_found)]}")


Valid IDs in chunks: [0, 1]
IDs cited by LLM:  [0, 1, 2]
HALLUCINATED citations: ['[ID:2]']
Valid citations: ['[ID:0]', '[ID:1]']


## Cell 10: split_codeblocks() - Protect Code Blocks


In [10]:
def split_codeblocks(text):
    """Split text into (segment, is_codeblock) pairs."""
    parts = re.split(r"(```)", text)
    result, is_code = [], False
    for part in parts:
        if part == "```":
            is_code = not is_code
            result.append((part, True))
        else:
            result.append((part, is_code))
    return result


t = "Text [ID:0].\n```python\ndef f(): pass\n```\nMore text."
for idx, (seg, is_code) in enumerate(split_codeblocks(t)):
    tag = "CODE" if is_code else "TEXT"
    print(f"  [{idx}] {tag}: {seg[:40]}...")


  [0] TEXT: Text [ID:0].
...
  [1] CODE: ```...
  [2] CODE: python
def f(): pass
...
  [3] CODE: ```...
  [4] TEXT: 
More text....


## Cell 11: split_sentences() - Multi-language Sentence Split


In [11]:
# Multi-language sentence split regex (Chinese, Arabic, Urdu, English)
SPLIT_RE = r"[^\|][；。？!！،؛؟۔\.\n]"


def split_sentences(text):
    """Split text into sentences using multi-language punctuation."""
    sents, last = [], 0
    for m in re.finditer(SPLIT_RE, text):
        end = m.end()
        s = text[last:end].strip()
        if s:
            sents.append(s)
        last = end
    rem = text[last:].strip()
    if rem:
        sents.append(rem)
    return sents


for t in ["Hello world. Second sentence!", "Test case? Yes."]:
    print(f"  {t!r}")
    print(f"    -> {split_sentences(t)}")


  'Hello world. Second sentence!'
    -> ['Hello world.', 'Second sentence!']
  'Test case? Yes.'
    -> ['Test case? Yes.']


## Cell 12: filter_noise() - Filter Short Sentences


In [12]:
MIN_LEN = 5


def filter_noise(sents):
    """Filter out sentences shorter than MIN_LEN characters."""
    out = []
    for s in sents:
        clean = re.sub(r'\[ID:\d+\]', '', s).strip()
        if len(clean) >= MIN_LEN:
            out.append(s)
        else:
            print(f"  Skip: {s!r}")
    return out


filtered = filter_noise([
    "Long sentence here [ID:0].",
    "Hi.", "OK",
    "Another valid one."
])
print(f"Kept: {len(filtered)}/4 sentences")


  Skip: 'Hi.'
  Skip: 'OK'
Kept: 2/4 sentences


## Cell 13: compute_similarity() - Hybrid Similarity


In [13]:
def cosine_sim(a, b):
    """Compute cosine similarity between two vectors."""
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0


def term_sim(t1, t2):
    """Compute Jaccard term similarity between two texts."""
    s1, s2 = set(t1.lower().split()), set(t2.lower().split())
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)


def hybrid_sim(sent, chunk_text, sv, cv, tw=0.1, vw=0.9):
    """Compute hybrid similarity = tw*term + vw*cosine."""
    return tw * term_sim(sent, chunk_text) + vw * cosine_sim(sv, cv)


# Test similarity computation
sv = emb.encode("Task Manager version control")[0]
cv0 = emb.encode(chunks[0]['content'])[0]
cv1 = emb.encode(chunks[1]['content'])[0]
uv = emb.encode("The weather is nice today")[0]

print("Similarity tests:")
print(f"  Related (chunk 0):    {hybrid_sim('Task Manager version control', chunks[0]['content'], sv, cv0):.4f}")
print(f"  Related (chunk 1):    {hybrid_sim('Task Manager version control', chunks[1]['content'], sv, cv1):.4f}")
print(f"  Unrelated:            {hybrid_sim('The weather is nice today', chunks[0]['content'], uv, cv0):.4f}")


Similarity tests:
  Related (chunk 0):    0.4533
  Related (chunk 1):    -0.0139
  Unrelated:            0.0796


## Cell 14: insert_citations() - Core Citation Injection


In [14]:
INIT_THRESH = 0.63
DECAY = 0.8
MIN_THRESH = 0.3
MAX_CITE = 4


def insert_citations(answer, chunks, emb_client):
    """Backend citation validation: split sentences, embed, match, inject [ID:x].

    Reproduces RAGFlow rag/nlp/search.py insert_citations() logic:
    1. Protect code blocks from modification
    2. Split into sentences (multi-language)
    3. Filter noise (short sentences)
    4. Embed each sentence
    5. Compute hybrid similarity with each chunk
    6. Threshold-based matching with decay (0.63 -> *0.8 -> min 0.3)
    7. Inject [ID:x] citation markers (max 4 per sentence)
    """
    parts = split_codeblocks(answer)
    cvecs = emb_client.encode([ch['content'] for ch in chunks])
    out_parts, all_refs = [], {}

    for seg, is_code in parts:
        if is_code:
            out_parts.append(seg)
            continue

        sents = split_sentences(seg)
        proc = []
        for sent in sents:
            clean = re.sub(r'\[ID:\d+\]', '', sent).strip()
            if len(clean) < MIN_LEN:
                proc.append(sent)
                continue

            sv = emb_client.encode(clean)[0]
            matches = []
            thresh = INIT_THRESH

            while thresh >= MIN_THRESH:
                for ci, ch in enumerate(chunks):
                    s = hybrid_sim(clean, ch['content'], sv, cvecs[ci])
                    if s >= thresh:
                        matches.append((ch['id'], s))
                if matches:
                    break
                thresh *= DECAY

            if matches:
                matches.sort(key=lambda x: -x[1])
                matches = matches[:MAX_CITE]
                sent = re.sub(r'\[ID:\d+\]', '', sent)
                tag = "".join(f"[ID:{cid}]" for cid, _ in matches)
                if sent and sent[-1] in '.!?;.,;\u3002\uFF01\uFF1F':
                    sent = sent.rstrip() + tag + sent[-1]
                else:
                    sent = sent + tag
                for cid, s in matches:
                    if cid not in all_refs or s > all_refs[cid]:
                        all_refs[cid] = s

            proc.append(sent)
        out_parts.append("".join(proc))

    final = "".join(out_parts)
    refs = []
    for cid, ms in sorted(all_refs.items()):
        ch = next(c for c in chunks if c['id'] == cid)
        refs.append({
            "id": cid,
            "doc_name": ch.get('doc_name', '?'),
            "page_num": ch.get('page_num'),
            "similarity": round(ms, 4)
        })
    return final, refs


print("insert_citations() function defined")
print(f"  Threshold: {INIT_THRESH} -> *{DECAY} -> min {MIN_THRESH}")
print(f"  Max citations per sentence: {MAX_CITE}")


insert_citations() function defined
  Threshold: 0.63 -> *0.8 -> min 0.3
  Max citations per sentence: 4


## Cell 15: Run Citation Validation


In [15]:
print("Running citation validation...")
fixed, refs = insert_citations(raw, chunks, emb)
print(f"\nCorrected answer ({len(fixed)} chars):")
print("-" * 60)
print(fixed)
print("-" * 60)
print(f"\nReferenced chunks ({len(refs)}):")
for r in refs:
    print(f"  [ID:{r['id']}] {r['doc_name'][:40]} "
          f"sim={r['similarity']:.4f} p.{r['page_num']}")


Running citation validation...

Corrected answer (767 chars):
------------------------------------------------------------
Based on the documentation provided, here is the answer:The IBM Content Navigator Task Manager does not support five key functions.[ID:0].These include document version control integration [ID:0] and custom workflow triggers based on document metadata changes.Additionally, real-time collaborative editing of task descriptions is not supported .[ID:0].The system also lacks automated task archiving based on retention policies , and cross-repository task delegation is not available.[ID:0].For FileNet deployments, administrators should note that the Content Engine handles document storage and metadata management , while the Process Engine manages workflow execution .[ID:1].Common issues include database connectivity problems and JNDI configuration errors.[ID:1].
------------------------------------------------------------

Referenced chunks (2):
  [ID:0] IBM_Content_Nav

## Cell 16: Build Structured JSON Response


In [16]:
def build_response(answer, refs, original=None):
    """Build standard RAGFlow-style API response."""
    resp = {
        "answer": answer,
        "reference_chunks": refs,
        "metadata": {
            "total_references": len(refs),
            "citation_format": "[ID:x]",
            "max_citations_per_sentence": MAX_CITE,
            "threshold_initial": INIT_THRESH,
            "threshold_min": MIN_THRESH,
            "tkweight": 0.1,
            "vtweight": 0.9
        }
    }
    if original:
        resp["original_answer"] = original
    return resp


resp = build_response(fixed, refs, raw)
print("Standard API Response:")
print("=" * 60)
out = json.dumps(resp, indent=2, ensure_ascii=False)
print(out[:1000])
if len(out) > 1000:
    print("... (truncated)")
print("=" * 60)


Standard API Response:
{
  "answer": "Based on the documentation provided, here is the answer:The IBM Content Navigator Task Manager does not support five key functions.[ID:0].These include document version control integration [ID:0] and custom workflow triggers based on document metadata changes.Additionally, real-time collaborative editing of task descriptions is not supported .[ID:0].The system also lacks automated task archiving based on retention policies , and cross-repository task delegation is not available.[ID:0].For FileNet deployments, administrators should note that the Content Engine handles document storage and metadata management , while the Process Engine manages workflow execution .[ID:1].Common issues include database connectivity problems and JNDI configuration errors.[ID:1].",
  "reference_chunks": [
    {
      "id": 0,
      "doc_name": "IBM_Content_Navigator_Administration_Guide.pdf",
      "page_num": 147,
      "similarity": 0.5355
    },
    {
      "id": 1,
 

## Cell 17: End-to-End Pipeline Function


In [17]:
def run_pipeline(question, chunks, llm_client, emb_client):
    """Complete RAGFlow citation processing pipeline."""
    print("=" * 60)
    print("RAGFlow Citation Processing Pipeline")
    print("=" * 60)

    # Phase 1
    sp, up = build_rag_prompt(chunks, question)
    print(f"1. Prompts built: sys={len(sp)} usr={len(up)} chars")

    # Phase 2
    raw_answer = llm_client.chat(sp, up)
    nc = len(re.findall(r'\[ID:(\d+)\]', raw_answer))
    print(f"2. LLM raw: {len(raw_answer)} chars, {nc} citations")

    # Phase 3
    fixed_answer, ref_list = insert_citations(raw_answer, chunks, emb_client)
    print(f"3. Validated: {len(fixed_answer)} chars, {len(ref_list)} refs")

    # Phase 4
    result = build_response(fixed_answer, ref_list, raw_answer)
    print(f"4. Response built with {len(ref_list)} references")
    print("=" * 60)
    return result


print("run_pipeline() function defined")


run_pipeline() function defined


## Cell 18: Run Full Pipeline


In [20]:
result = run_pipeline(Q, chunks, llm, emb)


RAGFlow Citation Processing Pipeline
1. Prompts built: sys=124 usr=2412 chars
2. LLM raw: 764 chars, 5 citations
3. Validated: 767 chars, 2 refs
4. Response built with 2 references


## Cell 19: Compare Raw vs Validated


In [21]:
orig_cites = set(re.findall(r'\[ID:(\d+)\]', result['original_answer']))
val_cites = set(re.findall(r'\[ID:(\d+)\]', result['answer']))
print("=" * 60)
print("Comparison: Raw LLM Output vs Backend Validated Output")
print("=" * 60)
print(f"Raw citations:      {sorted(orig_cites)}")
print(f"Validated citations: {sorted(val_cites)}")

removed = orig_cites - val_cites
added = val_cites - orig_cites

if removed:
    print(f"\nEliminated hallucinations: {sorted(removed)}")
else:
    print("\nNo hallucinated citations eliminated")

if added:
    print(f"Added missing citations:   {sorted(added)}")
else:
    print("No missing citations added")

print("\n--- Raw Answer ---")
print(result['original_answer'][:300])
print("\n--- Validated Answer ---")
print(result['answer'][:300])


Comparison: Raw LLM Output vs Backend Validated Output
Raw citations:      ['0', '1', '2']
Validated citations: ['0', '1']

Eliminated hallucinations: ['2']
No missing citations added

--- Raw Answer ---
Based on the documentation provided, here is the answer:

The IBM Content Navigator Task Manager does not support five key functions. These include document version control integration [ID:0] and custom workflow triggers based on document metadata changes. Additionally, real-time collaborative editi

--- Validated Answer ---
Based on the documentation provided, here is the answer:The IBM Content Navigator Task Manager does not support five key functions.[ID:0].These include document version control integration [ID:0] and custom workflow triggers based on document metadata changes.Additionally, real-time collaborative ed


## Cell 20: Multi-language (Chinese) Test


In [22]:
zh_text = (
    "根据文档，IBM Content Navigator Task Manager 不支持版本控制集成。"
    "此外还缺少基于元数据的工作流触发器。"
    "FileNet Content Engine 处理文档存储和元数据管理。"
)
print("Chinese test input:")
print(zh_text)
print()
zfixed, zrefs = insert_citations(zh_text, chunks, emb)
print(f"Corrected: {zfixed}")
print(f"References: {zrefs}")


Chinese test input:
根据文档，IBM Content Navigator Task Manager 不支持版本控制集成。此外还缺少基于元数据的工作流触发器。FileNet Content Engine 处理文档存储和元数据管理。

Corrected: 根据文档，IBM Content Navigator Task Manager 不支持版本控制集成。[ID:0]。此外还缺少基于元数据的工作流触发器。FileNet Content Engine 处理文档存储和元数据管理。
References: [{'id': 0, 'doc_name': 'IBM_Content_Navigator_Administration_Guide.pdf', 'page_num': 147, 'similarity': 0.4651}]
